# Train DINOv3 Detector

In [ ]:
%reload_ext autoreload
%autoreload 2

import gc
import json
import os
import random
import sys
from pathlib import Path

CURRENT = Path.cwd().resolve()
configured_root = os.environ.get("CODING_PROJECT_ROOT")
if configured_root:
    PROJECT_ROOT = Path(configured_root).expanduser().resolve()
else:
    candidates = [CURRENT, CURRENT.parent]
    PROJECT_ROOT = next(
        (path for path in candidates if (path / "pyproject.toml").exists()),
        CURRENT,
    )

%cd {PROJECT_ROOT}

os.environ["HF_HOME"] = str(PROJECT_ROOT / "cache" / "hf")
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "cache" / "matplotlib"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm

from Data_Pipeline.image_io import load_image, load_image_safe, resolve_path
from Data_Pipeline.manifest import Blocklist, load as load_manifest, validate
from Evaluation.metrics import compute
from Evaluation.visualization import reliability_diagram, use_house_style
from Modeling.common.device import device_report, get_device

use_house_style()


def progress(percent: float, stage: str, detail: str = "") -> None:
    """One consistent progress line for every notebook stage."""
    suffix = f" — {detail}" if detail else ""
    print(f"[PIPELINE {percent:6.1f}%] {stage}{suffix}", flush=True)


progress(2, "Setting", "project imports and autoreload are ready")

In [ ]:
TEST_MODE = False
MODEL_MODE = "saved"
RANDOM_SEED = 1337

SMOKE_TRAIN_PER_CLASS = 24
SMOKE_VALIDATION_PER_CLASS = 8
SMOKE_TEST_PER_CLASS = 8

ALLOW_BRANCH_B_FALLBACK = False

# Fixed before test evaluation to avoid leakage.
BRANCH_A_WEIGHT = 0.70
BRANCH_B_WEIGHT = 0.30

MIN_CONVERGENCE_EPOCHS = 3
TARGET_ROBUST_AUC = 0.85
MAX_GENERALIZATION_GAP = 0.10
TARGET_TEST_CORRECT_PERCENT = 80.0
TARGET_TPR_AT_1_FPR_PERCENT = 30.0
MAX_VALIDATION_ECE = 0.10

assert MODEL_MODE in {"train", "saved"}
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = get_device()
RUN_NAME = "notebook_smoke" if TEST_MODE else "notebook_dinov3"
WEIGHTS_DIR = PROJECT_ROOT / "weights" / "dinov3" / RUN_NAME
CHECKPOINT_PATH = WEIGHTS_DIR / "best_robust.pt"
LAST_CHECKPOINT_PATH = WEIGHTS_DIR / "last.pt"
CALIBRATION_PATH = PROJECT_ROOT / "weights" / "fusion" / f"{RUN_NAME}_temperature.json"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "notebook" / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

settings = pd.DataFrame(
    {
        "setting": [
            "TEST_MODE", "MODEL_MODE", "device", "run name",
            "Branch A checkpoint", "calibration", "output directory",
            "progress refresh", "minimum convergence epochs",
        ],
        "value": [
            TEST_MODE, MODEL_MODE, str(DEVICE), RUN_NAME,
            str(CHECKPOINT_PATH), str(CALIBRATION_PATH), str(OUTPUT_DIR),
            "every 5 minutes (or immediately at 100%)", MIN_CONVERGENCE_EPOCHS,
        ],
    }
)
display(settings)
display(pd.DataFrame([device_report()]))
progress(5, "Setting", "user controls validated")

In [ ]:
pretrained_status = pd.DataFrame(
    [
        {
            "asset": "Our Branch A checkpoint",
            "expected location": str(CHECKPOINT_PATH.relative_to(PROJECT_ROOT)),
            "available": CHECKPOINT_PATH.exists(),
        },
        {
            "asset": "Hugging Face cache",
            "expected location": str(Path(os.environ["HF_HOME"]).relative_to(PROJECT_ROOT)),
            "available": Path(os.environ["HF_HOME"]).exists(),
        },
    ]
)
display(pretrained_status)

In [ ]:
progress(8, "Data Input", "loading train, validation, benchmark, and test manifests")

MANIFEST_PATHS = {
    "train": PROJECT_ROOT / "data/train/manifest.csv",
    "validation": PROJECT_ROOT / "data/evaluation/validation_manifest.csv",
    "benchmark": PROJECT_ROOT / "data/evaluation/benchmark_manifest.csv",
    "test": PROJECT_ROOT / "data/test/manifest.csv",
}

official = {name: load_manifest(path) for name, path in MANIFEST_PATHS.items()}

for name, frame in official.items():
    print(f"\n{name.upper()} — {len(frame):,} rows")
    display(frame.head(5))

progress(12, "Data Input", "official manifest head(5) tables displayed")

In [ ]:
def balanced_sample(frame: pd.DataFrame, per_class: int, seed: int) -> pd.DataFrame:
    pieces = []
    for label, group in frame.groupby("label"):
        pieces.append(group.sample(min(per_class, len(group)), random_state=seed + int(label)))
    return pd.concat(pieces, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)


if TEST_MODE:
    active_train = balanced_sample(official["train"], SMOKE_TRAIN_PER_CLASS, RANDOM_SEED)
    active_validation = balanced_sample(
        official["validation"], SMOKE_VALIDATION_PER_CLASS, RANDOM_SEED + 10
    )
    active_test = balanced_sample(official["test"], SMOKE_TEST_PER_CLASS, RANDOM_SEED + 20)
    notebook_cache = PROJECT_ROOT / "cache/notebook/manifests"
    notebook_cache.mkdir(parents=True, exist_ok=True)
    ACTIVE_TRAIN_MANIFEST = notebook_cache / "train_smoke.csv"
    active_train.assign(split="train").to_csv(ACTIVE_TRAIN_MANIFEST, index=False)
else:
    active_train = official["train"].copy()
    active_validation = official["validation"].copy()
    active_test = official["test"].copy()
    ACTIVE_TRAIN_MANIFEST = MANIFEST_PATHS["train"]

active_counts = pd.concat(
    [
        frame.groupby("label").size().rename(name)
        for name, frame in {
            "train": active_train,
            "validation": active_validation,
            "test": active_test,
        }.items()
    ],
    axis=1,
).fillna(0).astype(int)
active_counts.index = active_counts.index.map({0: "authentic (0)", 1: "AI-generated (1)"})

print(f"Mode: {'SMALL END-TO-END TEST' if TEST_MODE else 'FULL DATA'}")
display(active_counts)
print("\nActive training rows:")
display(active_train.head(5))
progress(18, "Data Input", f"active train={len(active_train):,}, val={len(active_validation):,}, test={len(active_test):,}")

In [ ]:
progress(20, "Data Preparation", "checking paths, duplicates, classes, and quarantine")

blocklist = Blocklist.load()


def preparation_report(name: str, frame: pd.DataFrame) -> dict[str, object]:
    paths = frame["path"].astype(str)
    return {
        "dataset": name,
        "rows": len(frame),
        "missing files": sum(not resolve_path(path).exists() for path in paths),
        "duplicate sha256": int(frame.duplicated("sha256").sum()),
        "exact benchmark collisions": int(frame["sha256"].isin(blocklist.sha256).sum()),
        "authentic": int((frame["label"] == 0).sum()),
        "AI-generated": int((frame["label"] == 1).sum()),
    }


reports = pd.DataFrame(
    [
        preparation_report("train", active_train),
        preparation_report("validation", active_validation),
        preparation_report("test", active_test),
    ]
)
display(reports)

validate(active_train, blocklist=blocklist, check_phash=False)
validate(active_validation, blocklist=blocklist, check_phash=False)
validate(active_test, blocklist=blocklist, check_phash=False)
progress(28, "Data Preparation", "cleaning and quarantine audit passed")

In [ ]:
from Modeling.common.degradations import Cell, apply_chain

review_cells = [
    ("clean", None),
    ("JPEG quality 50", Cell("jpeg", 50)),
    ("blur sigma 1", Cell("blur", 1.0)),
    ("noise sigma .05", Cell("noise", 0.05)),
    ("center crop 80%", Cell("crop", 0.80)),
]

review_rows = pd.concat(
    [active_train[active_train["label"] == label].head(1) for label in (0, 1)],
    ignore_index=True,
)

fig, axes = plt.subplots(len(review_rows), len(review_cells), figsize=(16, 6), squeeze=False)
for row_index, row in review_rows.iterrows():
    original = load_image(row["path"])
    for column_index, (title, cell) in enumerate(review_cells):
        image = original if cell is None else apply_chain(
            original, [cell], rng=np.random.default_rng(RANDOM_SEED)
        )
        axes[row_index, column_index].imshow(image)
        axes[row_index, column_index].set_title(
            f"{title}\nlabel={int(row['label'])}", fontsize=10
        )
        axes[row_index, column_index].axis("off")
fig.suptitle(
    "Transformation review — authentic row first, AI-generated row second",
    fontsize=14,
)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "transformation_review.png", dpi=150, bbox_inches="tight")
plt.show()
progress(35, "Data Preparation", "sample transformations reviewed and saved")

In [ ]:
progress(38, "Data Modelling", "building Branch A configuration")

from Data_Pipeline.datasets import DataConfig
from Modeling.dinov3.backbone import BackboneConfig
from Modeling.dinov3.detector import Expert, ExpertConfig
from Modeling.dinov3.losses import LossWeights
from Modeling.dinov3.trainer import TrainConfig, train

if TEST_MODE:
    expert_config = ExpertConfig(
        backbone=BackboneConfig(
            name="vit_tiny_patch16_224",
            pretrained=False,
            pooling="mean",
            freeze=True,
            img_size=64,
        ),
        head_hidden=64,
        feature_correction=True,
        input_size=64,
        crop_mode="random_crop",
    )
    epochs, batch_size, workers = 50, 8, 0
    max_eval_images, n_inference_crops = len(active_validation), 2
else:
    expert_config = ExpertConfig(
        backbone=BackboneConfig(
            name="dinov3_l",
            pretrained=True,
            pooling="attention",
            freeze=True,
            lora=False,
        ),
        head_hidden=512,
        feature_correction=True,
        input_size=224,
        crop_mode="random_crop",
    )
    epochs, batch_size, workers = 10, 32, 4
    max_eval_images, n_inference_crops = 2000, 8

loss_weights = LossWeights(
    focal_gamma=2.0,
    focal_alpha=0.5,
    cls_degraded=1.0,
    kl=0.5,
    feat_mse=0.25,
)

data_config = DataConfig(
    manifest=str(ACTIVE_TRAIN_MANIFEST),
    input_size=expert_config.input_size,
    crop_mode=expert_config.crop_mode,
    pairwise=True,
    schedule="escalate",
    total_epochs=epochs,
    hflip=True,
    seed=RANDOM_SEED,
)

train_config = TrainConfig(
    epochs=epochs,
    batch_size=batch_size,
    lr=3e-4,
    weight_decay=0.02,
    warmup_epochs=0.5,
    amp=True,
    ema=False,
    swa=not TEST_MODE,
    swa_start_frac=0.7,
    num_workers=workers,
    seed=RANDOM_SEED,
    out_dir=str(PROJECT_ROOT / "outputs/runs" / RUN_NAME),
    weights_dir=str(WEIGHTS_DIR),
    progress_interval_seconds=300.0,
    eval_every=1,
    max_eval_images=max_eval_images,
    loss=loss_weights,
)

model_plan = pd.DataFrame(
    {
        "item": ["backbone", "pretrained", "input", "epochs", "batch", "train rows"],
        "value": [
            expert_config.backbone.name,
            expert_config.backbone.pretrained,
            expert_config.input_size,
            epochs,
            batch_size,
            len(active_train),
        ],
    }
)
display(model_plan)
progress(42, "Data Modelling", "Branch A configuration ready")

In [ ]:
training_result = None

if MODEL_MODE == "train":
    progress(45, "Branch A", "training started")
    branch_a_model = Expert(expert_config)
    print(branch_a_model.describe())
    training_result = train(
        branch_a_model,
        data_config,
        train_config,
        val_df=active_validation,
    )
    del branch_a_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError(f"training ended without {CHECKPOINT_PATH}")
    progress(60, "Branch A", f"training complete; saved {CHECKPOINT_PATH.name}")
else:
    progress(45, "Branch A", "saved-parameter mode selected")
    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError(
            f"Missing {CHECKPOINT_PATH}. Change MODEL_MODE='train', or copy the "
            "trained weights into this location."
        )
    progress(60, "Branch A", f"using saved checkpoint {CHECKPOINT_PATH}")

In [ ]:
from Modeling.dinov3.scorer import DINOv3Scorer
from Modeling.registry import check_param_budget, get

branch_a_scorer = DINOv3Scorer(
    checkpoint=str(CHECKPOINT_PATH),
    device=str(DEVICE),
    n_crops=n_inference_crops,
    seed=RANDOM_SEED,
)
branch_a_params = check_param_budget(branch_a_scorer)
print(f"Branch A ready: {branch_a_params:,} inference parameters")

In [ ]:
progress(62, "Branch B", "loading Community Forensics released checkpoint")

branch_b_name = "community_forensics"
try:
    branch_b_scorer = get(branch_b_name, device=str(DEVICE))
except Exception as error:
    if not ALLOW_BRANCH_B_FALLBACK:
        raise
    branch_b_name = "mean_intensity_smoke_fallback"
    branch_b_scorer = get("mean_intensity")
    print(
        "WARNING: Community Forensics could not load. "
        f"Smoke fallback enabled ({type(error).__name__}: {error})"
    )

branch_b_params = check_param_budget(branch_b_scorer)
print(f"Branch B ready: {branch_b_name}, {branch_b_params:,} inference parameters")
progress(66, "Branch B", "released branch ready")

In [ ]:
def score_frame(
    scorer,
    frame: pd.DataFrame,
    label: str,
    batch_size: int,
) -> np.ndarray:
    scores = np.full(len(frame), np.nan, dtype=np.float64)
    paths = frame["path"].astype(str).tolist()
    starts = range(0, len(paths), batch_size)
    for start in tqdm(
        starts,
        total=(len(paths) + batch_size - 1) // batch_size,
        desc=label,
        unit="batch",
        mininterval=300.0,
        maxinterval=300.0,
        miniters=0,
        dynamic_ncols=True,
    ):
        chunk_paths = paths[start : start + batch_size]
        images, positions = [], []
        for offset, path in enumerate(chunk_paths):
            image, error = load_image_safe(path)
            if image is None:
                print(f"WARNING: skipped unreadable validation image: {error}")
                continue
            images.append(image)
            positions.append(start + offset)
        if images:
            scores[positions] = scorer.score_checked(images)
    good = int(np.isfinite(scores).sum())
    print(f"[{label} 100.0%] scored {good:,}/{len(frame):,}", flush=True)
    return scores


progress(68, "Validation scoring", "Branch A")
validation_a = score_frame(branch_a_scorer, active_validation, "validation/branch-A", 8 if TEST_MODE else 32)
progress(71, "Validation scoring", "Branch B")
validation_b = score_frame(branch_b_scorer, active_validation, "validation/branch-B", 8 if TEST_MODE else 32)

In [ ]:
from Modeling.ensemble.calibration import TemperatureCalibrator

valid_mask = np.isfinite(validation_a) & np.isfinite(validation_b)
validation_raw_fused = (
    BRANCH_A_WEIGHT * validation_a[valid_mask]
    + BRANCH_B_WEIGHT * validation_b[valid_mask]
) / (BRANCH_A_WEIGHT + BRANCH_B_WEIGHT)
validation_labels = active_validation.loc[valid_mask, "label"].to_numpy(dtype=int)

if MODEL_MODE == "saved" and CALIBRATION_PATH.exists():
    calibrator = TemperatureCalibrator.load(CALIBRATION_PATH)
    calibration_action = "loaded"
else:
    calibrator = TemperatureCalibrator().fit(validation_raw_fused, validation_labels)
    calibrator.save(CALIBRATION_PATH)
    calibration_action = "fitted on validation"

validation_calibrated = calibrator.transform(validation_raw_fused)
print(
    f"Calibration {calibration_action}: T={calibrator.temperature:.6f} "
    f"-> {CALIBRATION_PATH}"
)
progress(77, "Fusion + calibration", "weights and temperature frozen before test")

In [ ]:
progress(79, "Test scoring", "Branch A")
test_a = score_frame(branch_a_scorer, active_test, "test/branch-A", 8 if TEST_MODE else 32)
progress(84, "Test scoring", "Branch B")
test_b = score_frame(branch_b_scorer, active_test, "test/branch-B", 8 if TEST_MODE else 32)

test_mask = np.isfinite(test_a) & np.isfinite(test_b)
results = active_test.loc[test_mask, ["path", "label", "source", "generator"]].copy()
results["branch_a"] = test_a[test_mask]
results["branch_b"] = test_b[test_mask]
results["fused_raw"] = (
    BRANCH_A_WEIGHT * results["branch_a"] + BRANCH_B_WEIGHT * results["branch_b"]
) / (BRANCH_A_WEIGHT + BRANCH_B_WEIGHT)
results["pred"] = calibrator.transform(results["fused_raw"].to_numpy())

metric_rows = []
labels_array = results["label"].to_numpy(dtype=int)
for model_name, column in (
    ("Branch A — our detector", "branch_a"),
    (f"Branch B — {branch_b_name}", "branch_b"),
    ("Fused + calibrated", "pred"),
):
    scores = results[column].to_numpy()
    predicted = (scores >= 0.5).astype(int)
    correct = predicted == labels_array
    authentic = labels_array == 0
    generated = labels_array == 1
    metric_rows.append({
        "model": model_name,
        **compute(labels_array, scores),
        "pred_correct_percent": 100.0 * correct.mean(),
        "authentic_correct_percent": 100.0 * correct[authentic].mean(),
        "ai_generated_correct_percent": 100.0 * correct[generated].mean(),
        "correct_predictions": int(correct.sum()),
        "total_predictions": int(len(correct)),
    })
metrics_table = pd.DataFrame(metric_rows)

results["predicted_label"] = (results["pred"] >= 0.5).astype(int)
results["correct"] = results["predicted_label"] == results["label"]
fused_correct = int(results["correct"].sum())
fused_correct_percent = 100.0 * fused_correct / max(1, len(results))

display(
    metrics_table[
        [
            "model", "pred_correct_percent", "authentic_correct_percent",
            "ai_generated_correct_percent", "roc_auc", "average_precision",
            "tpr_at_fpr0.01", "tpr_at_fpr0.05",
        ]
    ].rename(columns={
        "pred_correct_percent": "Pred correct (%)",
        "authentic_correct_percent": "Authentic correct (%)",
        "ai_generated_correct_percent": "AI-generated correct (%)",
    }).round(2)
)
print(
    f"Final fused predictions correct: {fused_correct:,}/{len(results):,} "
    f"= {fused_correct_percent:.2f}%"
)
display(results.head(5))

csv_path = OUTPUT_DIR / "test_predictions_with_labels.csv"
json_path = OUTPUT_DIR / "predictions.json"
metrics_path = OUTPUT_DIR / "metrics.csv"
results.to_csv(csv_path, index=False)
metrics_table.to_csv(metrics_path, index=False)
json_records = [
    {"image_path": path, "pred": float(prediction)}
    for path, prediction in zip(results["path"], results["pred"])
]
json_path.write_text(json.dumps(json_records, indent=2) + "\n")
progress(90, "Data Modelling", f"predictions and metrics saved under {OUTPUT_DIR}")

In [ ]:
history = training_result["history"] if training_result is not None else []
if not history and LAST_CHECKPOINT_PATH.exists():
    history = torch.load(LAST_CHECKPOINT_PATH, map_location="cpu").get("history", [])

history_rows = []
for entry in history:
    row = {}
    row.update(entry.get("train", {}))
    row.update(entry.get("summary", {}))
    history_rows.append(row)
history_frame = pd.DataFrame(history_rows)

loss_columns = [
    column for column in (
        "train_total", "train_cls_clean", "train_cls_degraded", "train_kl", "train_feat_mse"
    )
    if column in history_frame and history_frame[column].notna().any()
]

if loss_columns:
    loss_milestones = (
        history_frame.assign(epoch=history_frame["epoch"] + 1)
        .loc[lambda frame: frame["epoch"] % 10 == 0, ["epoch", *loss_columns]]
        .copy()
    )
    display(loss_milestones.round(6))

    if loss_milestones.empty:
        print("No 10-epoch milestone is available yet. Complete epoch 10 first.")
    else:
        fig, ax = plt.subplots(figsize=(9, 5))
        for column in loss_columns:
            ax.plot(
                loss_milestones["epoch"],
                loss_milestones[column],
                marker="o",
                label=column.removeprefix("train_").replace("_", " "),
            )
        ax.set_title("Training convergence — 10-epoch milestones")
        ax.set_xlabel("Epoch (shown every 10 epochs)")
        ax.set_ylabel("Average loss (lower is better)")
        ax.set_xticks(loss_milestones["epoch"].astype(int).tolist())
        ax.legend(title="Loss component")
        fig.tight_layout()
        fig.savefig(OUTPUT_DIR / "training_loss_convergence.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    print("This checkpoint has no persisted epoch-loss history.")
progress(92, "Visualization", "training-loss statistics reviewed")

In [ ]:
auc_columns = [
    column for column in ("clean_auc", "robust_auc", "worst_cell_auc")
    if column in history_frame and history_frame[column].notna().any()
]
if auc_columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for column in auc_columns:
        axes[0].plot(
            history_frame["epoch"] + 1,
            history_frame[column],
            marker="o",
            label=column.replace("_", " "),
        )
    axes[0].axhline(
        TARGET_ROBUST_AUC, color="#16a34a", linestyle="--", label="robust AUC target"
    )
    axes[0].set_title("Branch A validation AUC through training")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("AUC (higher is better)")
    axes[0].set_ylim(0, 1.02)
    axes[0].legend()

    axes[1].plot(
        history_frame["epoch"] + 1, history_frame["gap"], marker="o", color="#dc2626"
    )
    axes[1].axhline(
        MAX_GENERALIZATION_GAP,
        color="#16a34a",
        linestyle="--",
        label="maximum target gap",
    )
    axes[1].axhline(0, color="#64748b", linewidth=1)
    axes[1].set_title("Clean-to-degraded generalisation gap")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Clean AUC - robust AUC (closer to 0 is better)")
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "validation_convergence.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No robust-validation history is available for this checkpoint.")
progress(94, "Visualization", "validation convergence and gap reviewed")

In [ ]:
correctness_plot = metrics_table.set_index("model")[[
    "pred_correct_percent", "authentic_correct_percent", "ai_generated_correct_percent"
]].rename(columns={
    "pred_correct_percent": "Overall",
    "authentic_correct_percent": "Authentic",
    "ai_generated_correct_percent": "AI-generated",
})
fig, ax = plt.subplots(figsize=(11, 5.5))
correctness_plot.plot(kind="bar", ax=ax, color=["#0f766e", "#3b82f6", "#f97316"])
ax.axhline(
    TARGET_TEST_CORRECT_PERCENT, color="#16a34a", linestyle="--", label="project target"
)
ax.set_title("Test predictions correct — overall and by ground-truth class")
ax.set_xlabel("Model output")
ax.set_ylabel("Correct predictions (%)")
ax.set_ylim(0, 102)
ax.tick_params(axis="x", rotation=10)
ax.legend(title="Correctness group", ncol=2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "test_percent_correct.png", dpi=150, bbox_inches="tight")
plt.show()
progress(96, "Visualization", "test correctness reviewed")

In [ ]:
_, ece_before = reliability_diagram(validation_labels, validation_raw_fused)
plt.suptitle("Validation reliability before temperature calibration", y=1.02)
plt.show()

_, ece_after = reliability_diagram(
    validation_labels,
    validation_calibrated,
    path=OUTPUT_DIR / "validation_reliability_after.png",
)
plt.suptitle("Validation reliability after temperature calibration", y=1.02)
plt.show()
print(f"Expected calibration error: {ece_before:.4f} -> {ece_after:.4f}")
progress(97, "Visualization", "calibration reliability reviewed")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)
score_rng = np.random.default_rng(RANDOM_SEED)
for ax, (title, column) in zip(
    axes,
    [
        ("Branch A score distribution", "branch_a"),
        ("Branch B score distribution", "branch_b"),
        ("Fused calibrated distribution", "pred"),
    ],
):
    class_scores = [
        results.loc[results["label"] == label, column].dropna().to_numpy()
        for label in (0, 1)
    ]
    boxes = ax.boxplot(
        class_scores,
        positions=[0, 1],
        orientation="horizontal",
        widths=0.42,
        patch_artist=True,
        showfliers=False,
        medianprops={"color": "#14191b", "linewidth": 2},
    )
    for patch, colour in zip(boxes["boxes"], ("#3b82f6", "#f97316")):
        patch.set_facecolor(colour)
        patch.set_alpha(0.28)
    for label, (values, colour) in enumerate(zip(class_scores, ("#3b82f6", "#f97316"))):
        shown = values if len(values) <= 500 else score_rng.choice(values, 500, replace=False)
        jitter = score_rng.normal(label, 0.055, size=len(shown))
        ax.scatter(shown, jitter, s=18, alpha=0.45, color=colour, edgecolors="none")
        if len(values):
            ax.text(
                float(np.median(values)), label + 0.28, f"median {np.median(values):.2f}",
                ha="center", va="bottom", fontsize=8, color=colour,
            )
    ax.axvline(0.5, color="#475569", linestyle="--", linewidth=1.2)
    ax.set_title(title)
    ax.set_xlabel("Predicted P(AI-generated)")
    ax.set_xlim(0, 1)
    ax.set_xticks(np.linspace(0, 1, 6))
axes[0].set_yticks([0, 1], labels=["Authentic", "AI-generated"])
fig.suptitle("Score separation — farther apart is better", fontsize=13)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "score_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

fused_metrics = metrics_table.loc[metrics_table["model"] == "Fused + calibrated"].iloc[0]
enough_epochs = len(history_frame) >= MIN_CONVERGENCE_EPOCHS
loss_decreased = (
    enough_epochs
    and "train_total" in history_frame
    and history_frame["train_total"].iloc[-1] < history_frame["train_total"].iloc[0]
)
best_robust_auc = history_frame.get("robust_auc", pd.Series(dtype=float)).max()
latest_gap = (
    float(history_frame["gap"].dropna().iloc[-1])
    if "gap" in history_frame and history_frame["gap"].notna().any()
    else float("nan")
)

readiness_rows = [
    {
        "quality question": "Did training loss decrease across enough epochs?",
        "observed": (
            f"{history_frame['train_total'].iloc[0]:.4f} -> "
            f"{history_frame['train_total'].iloc[-1]:.4f}"
            if "train_total" in history_frame and not history_frame.empty else "unavailable"
        ),
        "target": f"decrease across >= {MIN_CONVERGENCE_EPOCHS} epochs",
        "status": "PASS" if loss_decreased else (
            "NOT ASSESSABLE" if not enough_epochs else "FAIL"
        ),
    },
    {
        "quality question": "Does Branch A work across degradations?",
        "observed": f"best robust AUC {best_robust_auc:.3f}",
        "target": f">= {TARGET_ROBUST_AUC:.2f}",
        "status": (
            "PASS"
            if pd.notna(best_robust_auc) and best_robust_auc >= TARGET_ROBUST_AUC
            else "FAIL"
        ),
    },
    {
        "quality question": "Is the clean-to-degraded gap controlled?",
        "observed": f"gap {latest_gap:.3f}",
        "target": f"<= {MAX_GENERALIZATION_GAP:.2f}",
        "status": (
            "PASS"
            if pd.notna(latest_gap) and latest_gap <= MAX_GENERALIZATION_GAP
            else "FAIL"
        ),
    },
    {
        "quality question": "Are enough test predictions correct?",
        "observed": f"{fused_metrics['pred_correct_percent']:.2f}%",
        "target": f">= {TARGET_TEST_CORRECT_PERCENT:.1f}%",
        "status": (
            "PASS"
            if fused_metrics["pred_correct_percent"] >= TARGET_TEST_CORRECT_PERCENT
            else "FAIL"
        ),
    },
    {
        "quality question": "Does it catch generated images at 1% FPR?",
        "observed": f"{100 * fused_metrics['tpr_at_fpr0.01']:.2f}%",
        "target": f">= {TARGET_TPR_AT_1_FPR_PERCENT:.1f}%",
        "status": (
            "PASS"
            if 100 * fused_metrics["tpr_at_fpr0.01"] >= TARGET_TPR_AT_1_FPR_PERCENT
            else "FAIL"
        ),
    },
    {
        "quality question": "Are validation probabilities calibrated?",
        "observed": f"ECE {ece_after:.3f}",
        "target": f"<= {MAX_VALIDATION_ECE:.2f}",
        "status": "PASS" if ece_after <= MAX_VALIDATION_ECE else "FAIL",
    },
]
readiness_table = pd.DataFrame(readiness_rows)
status_colours = {"PASS": "#0E9384", "FAIL": "#A33A3E", "NOT ASSESSABLE": "#9A6407"}
fig, ax = plt.subplots(figsize=(12, 6.2))
ax.set_xlim(0, 1)
ax.set_ylim(-0.7, len(readiness_table) - 0.3)
ax.axis("off")
for row_index, row in readiness_table.iloc[::-1].reset_index(drop=True).iterrows():
    display_status = "NOT ASSESSABLE" if TEST_MODE else row["status"]
    colour = status_colours[display_status]
    ax.scatter(0.035, row_index, s=260, color=colour, clip_on=False)
    ax.text(0.035, row_index, "✓" if display_status == "PASS" else "!",
            ha="center", va="center", color="white", fontweight="bold")
    ax.text(0.075, row_index + 0.13, row["quality question"],
            ha="left", va="center", fontsize=10, fontweight="bold")
    ax.text(0.075, row_index - 0.16, f"Observed: {row['observed']}   •   Target: {row['target']}",
            ha="left", va="center", fontsize=9, color="#4A5459")
    ax.text(0.97, row_index, display_status, ha="right", va="center",
            fontsize=9, fontweight="bold", color=colour)
    if row_index < len(readiness_table) - 1:
        ax.hlines(row_index + 0.5, 0.075, 0.97, color="#E4E9E9", linewidth=1)
ax.set_title("Model-readiness checklist", loc="left", fontsize=14, pad=14)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "model_readiness.png", dpi=150, bbox_inches="tight")
plt.show()

if TEST_MODE:
    readiness_verdict = (
        "PIPELINE VALIDATED — quality/convergence not assessable from smoke data"
    )
elif (readiness_table["status"] == "PASS").all():
    readiness_verdict = "READY CANDIDATE — all configured quality gates passed"
else:
    readiness_verdict = (
        "NOT YET GOOD ENOUGH — review failed quality gates before release"
    )
print(f"\nMODEL READINESS: {readiness_verdict}")
progress(99, "Visualization", "convergence and readiness assessment complete")

In [ ]:
summary = {
    "test_mode": TEST_MODE,
    "model_mode": MODEL_MODE,
    "device": str(DEVICE),
    "branch_a_checkpoint": str(CHECKPOINT_PATH),
    "branch_b": branch_b_name,
    "temperature": calibrator.temperature,
    "test_images_scored": len(results),
    "test_predictions_correct": f"{fused_correct}/{len(results)}",
    "test_predictions_correct_percent": round(fused_correct_percent, 2),
    "model_readiness": readiness_verdict,
    "predictions_json": str(json_path),
    "metrics_csv": str(metrics_path),
    "output_directory": str(OUTPUT_DIR),
}
(OUTPUT_DIR / "run_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
display(pd.DataFrame(summary.items(), columns=["item", "value"]))
progress(100, "Complete", "full notebook workflow finished successfully")